[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/t06_diffeq.ipynb)

# How long does soup take to cool? Differential equations: name the type, then solve

Soup at 90 °C in a 20 °C room is at 60 °C after ten minutes. "Three degrees a minute" says it reaches 40 °C at about
17 minutes. It's wrong: the rate shrinks as the gap to the room shrinks, and the soup takes about **22.4 minutes**.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top. In a check cell, type the equation and your answer, then
press **Check**: it puts your answer back into the equation and says whether both sides match. When a slider
appears, drag it. It works in your browser (no install) and in Google Colab. Nothing can break: if something looks
odd, run everything again from the top.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and the tools. Run this box first.
import re
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, FloatSlider, Text, Button, Output, VBox, HBox, Layout
from IPython.display import display
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # solution curves · rates and slopes · integrated pieces

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})
plt.close(plt.figure())                     # warms up the plotting: the first figure is slow in the browser

x, t = sp.symbols("x t")
K_SOUP = sp.log(sp.Rational(7, 4)) / 10          # the soup's k: the gap is multiplied by 4/7 every ten minutes
kk = sp.Symbol("k", positive=True)               # typed as k; its value is put in only to test
NAMES = {"x": x, "t": t, "e": sp.E, "ln": sp.log, "log": sp.log, "pi": sp.pi, "sqrt": sp.sqrt, "k": kk}
RULES = standard_transformations + (implicit_multiplication_application, convert_xor)

def read(s, extra=None):
    """Turn what you type into maths: x^2, e^(2x), ln(x), sin(x); k is the soup's cooling constant."""
    return parse_expr(s.replace("−", "-"), local_dict={**NAMES, **(extra or {})}, transformations=RULES)

def say(e):
    return str(e).replace("**", "^").replace("*", "·").replace("log(", "ln(").replace("exp(", "e^(")

def put_back(equation, answer):
    """Put the answer back into the equation. The equation is written with a prime for the derivative, like
    y' = 2*y or T' = -k*(T - 20) (dy/dx also works). Returns (dependent, independent, left side, right side)."""
    eq = equation.replace("−", "-")
    m = re.search(r"d(\w)/d(\w)", eq)
    dep = m.group(1) if m else re.search(r"([A-Za-z])'", eq).group(1)
    ind = m.group(2) if m else ("t" if dep == "T" else "x")
    eq = re.sub(rf"d{dep}/d{ind}", "D_", eq).replace(f"{dep}'", "D_")
    lhs, rhs = eq.split("=")
    iv = NAMES[ind]
    Y = read(answer)
    sub = {"D_": sp.diff(Y, iv), dep: Y}
    return dep, ind, sp.simplify(read(lhs, sub)), sp.simplify(read(rhs, sub))

def same(L, R):
    return sp.simplify((L - R).subs(kk, K_SOUP)) == 0

def check(equation, answer):
    dep, ind, L, R = put_back(equation, answer)
    print(f"{equation}   with   {dep} = {answer}")
    print(f"  left side:   {say(L)}")
    print(f"  right side:  {say(R)}")
    if same(L, R): print("  ✓ both sides match: it solves the equation")
    else:
        iv = NAMES[ind]; at = 10 if ind == "t" else 1
        l, r = (float(e.subs(kk, K_SOUP).subs(iv, at)) for e in (L, R))
        print(f"  ✗ they do not match: at {ind} = {at}, the left side is {l:.3f} and the right side is {r:.3f}")

def checker(equation="", answer=""):
    """Two boxes (the equation, your answer) and a Check button."""
    be = Text(value=equation, placeholder="y' = 2*y", description="equation", layout=Layout(width="330px"))
    ba = Text(value=answer, placeholder="your answer", description="answer", layout=Layout(width="330px"))
    button = Button(description="Check")
    out = Output()
    def on_check(_):
        out.clear_output()
        with out:
            try: check(be.value, ba.value)
            except Exception as err: print("I couldn't read that. Try e.g. y' = 2*y and 5*e^(2*x)  (", err, ")")
    button.on_click(on_check)
    display(VBox([HBox([be, ba, button]), out]))

def slope_field(ax, f, xs, ys, scale=0.6, color=SAGE):
    """Short ticks with slope f(x, y) at each grid point: every solution curve runs along them."""
    for X in xs:
        for Yv in ys:
            s = f(X, Yv)
            dx = scale / np.sqrt(1 + s * s) * (xs[1] - xs[0])
            ax.plot([X - dx / 2, X + dx / 2], [Yv - s * dx / 2, Yv + s * dx / 2], color=color, lw=1.2, alpha=0.7)

## 1 · The steady guess against the real curve

Newton's rule: dT/dt = −k(T − 20). The rate is proportional to the gap between the soup and the room. Here
k = ln(7/4) ÷ 10, so the gap of 70 is multiplied by 4/7 every ten minutes.

In [ ]:
k = float(K_SOUP)
ts = np.linspace(0, 40, 400)
T = 20 + 70 * np.exp(-k * ts)
t40 = np.log(3.5) / k
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.plot(ts, T, color=TEAL, lw=3, label="the real curve: 20 + 70 e^(−kt)")
ax.plot(ts[ts <= 25], 90 - 3 * ts[ts <= 25], color=CHALK, lw=1.5, ls="--", label="the steady guess: 90 − 3t")
ax.axhline(20, color=DIM, ls=":", label="the room: 20 °C"); ax.axhline(40, color=DIM, lw=0.8)
ax.plot([t40], [40], "o", color=CORAL, ms=10)
ax.set_xlabel("minutes"); ax.set_ylabel("°C"); ax.set_ylim(0, 95); ax.legend(loc="upper right")
ax.set_title(f"40 °C at {t40:.1f} minutes, not 16.7")
plt.show()
print(f"after 10 min: {20 + 70 * np.exp(-10 * k):.1f} °C · after 20 min: {20 + 70 * np.exp(-20 * k):.1f} °C · "
      f"40 °C at t = ln(7/2) ÷ ln(7/4) × 10 = {t40:.2f} min")

**Notice.** The guess and the curve agree at 0 and at 10 minutes, then part. Each ten minutes takes off less than the
ten before: 30 degrees, then about 17, then about 10.

## 2 · General and particular solutions

dy/dx = e^(x − y) sorts into e^y dy = e^x dx, so e^y = e^x + C. Every C gives a curve: the general solution. Move C,
and see which curve passes through the start y(0) = ln 2: that one is the particular solution, C = 1.

In [ ]:
def family(C=1.0):
    xs = np.linspace(-2, 2, 300)
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    slope_field(ax, lambda X, Y: np.exp(X - Y), np.linspace(-2, 2, 17), np.linspace(-2, 3, 13))
    for c in (-0.5, 0.5, 1.5, 2.5, 4.0):
        ok = np.exp(xs) + c > 0
        ax.plot(xs[ok], np.log(np.exp(xs[ok]) + c), color=DIM, lw=1)
    ok = np.exp(xs) + C > 0
    ax.plot(xs[ok], np.log(np.exp(xs[ok]) + C), color=TEAL, lw=3, label=f"e^y = e^x + {C:.1f}")
    ax.plot([0], [np.log(2)], "o", color=CORAL, ms=10, label="the start: y(0) = ln 2")
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 3); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(loc="upper left")
    ax.set_title("the start picks one curve" + ("  ✓" if abs(C - 1) < 1e-9 else ""))
    plt.show()

interact(family, C=FloatSlider(value=0.5, min=-0.5, max=4.0, step=0.25, description="C"));

**Notice.** The green ticks are the slopes the equation demands. Every curve in the family runs along them. Only
C = 1 passes through the coral start.

## 3 · The video's answers, put back

Each answer is tested the honest way: put it back into its equation.

In [ ]:
for eq, ans in [("y' = e^(x - y)", "ln(e^x + 1)"), ("y' = (x + y)/x", "x*ln(x) + 3*x"),
                ("y' = (x^2 + y^2)/(2*x*y)", "sqrt(x^2 - 3*x/2)"), ("y' + 2*y = 4*x", "2*x - 1 + 5*e^(-2*x)"),
                ("y' + y/x = x^2", "x^3/4"), ("y' = 2*y", "5*e^(2*x)"), ("y' = 2*y", "e^(2*x) + 4")]:
    dep, ind, L, R = put_back(eq, ans)
    print(f"{eq:<26} y = {ans:<22} " + ("✓ matches" if same(L, R) else f"✗ off by {say(sp.simplify(L - R))}"))

## 4 · Check your answer

Type the equation with a prime for the derivative (`y' = 2*y`; `dy/dx` works too) and your answer for y. Use
`e^(2*x)` for e to the 2x, `ln(x)`, `x^2`. Then press **Check**.

In [ ]:
# Two boxes and a button: type the equation and your answer, then press Check.
checker()

## 5 · The soup, and the room

Drag the room's temperature. The soup starts at 90 °C with the same k. The green ticks are the rates Newton's rule
demands; the curve bends towards the new dashed line, however warm or cold the room.

In [ ]:
def soup(room=20.0):
    ts = np.linspace(0, 60, 400)
    fig, ax = plt.subplots(figsize=(9, 4.4))
    slope_field(ax, lambda X, Y: -k * (Y - room), np.linspace(0, 60, 16), np.linspace(0, 100, 11), 0.8)
    ax.plot(ts, room + (90 - room) * np.exp(-k * ts), color=TEAL, lw=3, label="the soup")
    ax.axhline(room, color=CORAL, ls="--", lw=2, label=f"the room: {room:.0f} °C")
    ax.set_xlim(0, 60); ax.set_ylim(0, 100); ax.set_xlabel("minutes"); ax.set_ylabel("°C"); ax.legend(loc="upper right")
    ax.set_title("dT/dt = −k (T − room)")
    plt.show()

interact(soup, room=FloatSlider(value=20, min=0, max=60, step=2, description="room (°C)"));

**Notice.** The curve never crosses the dashed line: as the gap closes, the rate closes with it.

Now try the steady-rate guess, T = 90 − 3t, in the soup's equation. (Here `k` is the soup's constant.)

In [ ]:
# The soup's equation is filled in: type an answer for T and press Check.
checker(equation="T' = -k*(T - 20)")

## The challenge
(1 + x²) dy/dx + 2xy = 4x², with y = 0 when x = 0. Hint: look at the left side before you reach for a factor. Check
your answer above with the equation `(1 + x^2)*y' + 2*x*y = 4*x^2`.

## Sources
I. Newton (published without his name), "Scala graduum caloris", *Philosophical Transactions* 22 (1701). CBSE
Mathematics 041 syllabus 2026–27 (Differential Equations). All problems are our own. Retrieved 2026-10-24.